# 06b — Random Forest

**Milestone 3 · Task 7.** Build the Random Forest classifier and record its
performance against the 54.7% majority-class floor.

A Random Forest builds many decision trees, each on a random subsample of
rows and considering a random subset of features at each split, then
averages their votes. Individual trees overfit badly; averaging many
decorrelated ones cancels most of that out.

**It assumes almost nothing** - no linearity, no scaling, no
distributional assumptions - and it finds interactions automatically. If
high goals are only damaging in certain categories, some tree will split
on it without being told to.

### How this fits with the other two notebooks

Three people, three models, three notebooks, run in parallel. Each notebook
writes its own CSV to `reports/`, and Task 8 concatenates all three into one
comparison table.

**Two rules so the results line up:**

1. Build `X` and `y` with `build_model_matrix()` — never by hand.
2. Report through `evaluate()` — never write your own metric code. It returns
   a one-row DataFrame so the rows from all three notebooks stack cleanly.

Do not edit anything in `src/`. If you think a shared module needs changing,
raise it with the repo owner first.

## Setup

Run this first. On Colab it clones the repo; locally it does nothing.

In [1]:
# Works both locally and on Google Colab.
# On Colab this clones the repo; locally it does nothing.
import sys, os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "State-Street-1B-kickstarter-crowdfunding-recommendation-engine"
    GITHUB_URL = f"https://github.com/YOUR-ORG/{REPO}.git"   # <- edit this
    if not os.path.exists(REPO):
        os.system(f"git clone -q {GITHUB_URL}")
    os.chdir(f"{REPO}/notebooks")

sys.path.insert(0, "..")
print("Colab" if IN_COLAB else "local", "| cwd:", Path.cwd().name)

local | cwd: notebooks


In [2]:
import pandas as pd
import numpy as np

from src.features import build_model_matrix
from src.modeling import make_models, align_columns, random_split, chronological_split
from src.evaluation import evaluate, majority_class_baseline, confusion_frame

pd.set_option("display.width", 220)
EXTRA = ["log_city_project_count", "log_goal_vs_subcat_median"]

# The parquet splits are gitignored, so they will not exist on a fresh clone.
# Rebuild them if missing -- about 40 seconds.
if not Path("../data/m3_random_train.parquet").exists():
    print("Splits not found. Rebuilding from the raw data ...")
    from src.loading import load_raw
    from src.cleaning import clean
    from src.features import build_features, add_location_frequency, add_relative_goal

    feat = build_features(clean(load_raw("../data/DSI_kickstarterscrape_dataset_csv.zip"),
                                verbose=False)[0])

    def _prep(tr, te):
        tr, city = add_location_frequency(tr)
        tr, goal = add_relative_goal(tr)
        te, _ = add_location_frequency(te, reference=city)
        te, _ = add_relative_goal(te, reference=goal)
        return tr, te

    for (a, b), names in [(_prep(*random_split(feat)), ("m3_random_train", "m3_random_test")),
                          (_prep(*chronological_split(feat)), ("m3_chrono_train", "m3_chrono_test"))]:
        a.to_parquet(f"../data/{names[0]}.parquet", index=False)
        b.to_parquet(f"../data/{names[1]}.parquet", index=False)
    print("Splits written.")

SPLITS = {
    "random": (pd.read_parquet("../data/m3_random_train.parquet"),
               pd.read_parquet("../data/m3_random_test.parquet")),
    "chronological": (pd.read_parquet("../data/m3_chrono_train.parquet"),
                      pd.read_parquet("../data/m3_chrono_test.parquet")),
}
for n, (tr, te) in SPLITS.items():
    print(f"{n:15s} train {len(tr):,}  test {len(te):,}")

random          train 30,702  test 7,676
chronological   train 30,702  test 7,676


## The floor

Predicting the majority class for every row. Any model that fails to beat this has learned nothing, which is why reporting accuracy without this number is misleading.

Note the recall of 1.0: always guessing "successful" catches every success and is wrong about every failure. The floor still scores 0.71 on F1, so F1 alone is not enough either.

In [3]:
tr, te = SPLITS["random"]
floor = majority_class_baseline(tr["success"], te["success"])
floor.round(4)

,model,feature_set,split,accuracy,precision,recall,f1,roc_auc
0,Majority class,—,—,0.5473,0.5473,1.0,0.7074,0.5


## Fit on all four combinations

Two feature sets x two splits. `launch_only` is primary — it excludes
`updates`, which is unavailable at campaign launch. `plus_campaign` adds it
back so the cost of that exclusion can be measured rather than asserted.

In [4]:
results = []

for split_name, (train, test) in SPLITS.items():
    for fs in ["launch_only", "plus_campaign"]:
        X_train, y_train = build_model_matrix(train, fs, extra_numeric=EXTRA)
        X_test, y_test = build_model_matrix(test, fs, extra_numeric=EXTRA)
        X_test = align_columns(X_train, X_test)

        model = make_models()["Random Forest"]
        model.fit(X_train, y_train)
        results.append(evaluate(model, X_test, y_test,
                                label="Random Forest", feature_set=fs, split=split_name))
        print(f"  fitted {fs:14s} {split_name}", flush=True)

res = pd.concat(results, ignore_index=True)
res.round(4)

  fitted launch_only    random


  fitted plus_campaign  random


  fitted launch_only    chronological


  fitted plus_campaign  chronological


,model,feature_set,split,accuracy,precision,recall,f1,roc_auc
0,Random Forest,launch_only,random,0.7153,0.7041,0.8277,0.7609,0.7798
1,Random Forest,plus_campaign,random,0.8180,0.8158,0.8622,0.8383,0.8972
2,Random Forest,launch_only,chronological,0.7200,0.7037,0.8232,0.7588,0.7906
3,Random Forest,plus_campaign,chronological,0.8056,0.8094,0.8327,0.8209,0.8898


## The primary configuration

`launch_only` on the `random` split is the one that answers the project's actual question.

In [5]:
primary = res[(res.feature_set == "launch_only") & (res.split == "random")].iloc[0]
floor_acc = float(floor["accuracy"].iloc[0])

print(f"Majority-class floor:  {floor_acc:.1%} accuracy, F1 {float(floor['f1'].iloc[0]):.3f}")
print(f"Random Forest: {primary.accuracy:.1%} accuracy, F1 {primary.f1:.3f}, "
      f"AUC {primary.roc_auc:.3f}")
print(f"Lift over floor:       {primary.accuracy - floor_acc:+.1%}")

Majority-class floor:  54.7% accuracy, F1 0.707
Random Forest: 71.5% accuracy, F1 0.761, AUC 0.780
Lift over floor:       +16.8%


## What `updates` is worth

The difference between the two feature sets is the price of excluding a variable that does not exist at launch.

In [6]:
a = res[(res.feature_set == "launch_only") & (res.split == "random")].accuracy.iloc[0]
b = res[(res.feature_set == "plus_campaign") & (res.split == "random")].accuracy.iloc[0]
print(f"launch-only   {a:.1%}")
print(f"with updates  {b:.1%}   ({b - a:+.1%})")

launch-only   71.5%
with updates  81.8%   (+10.3%)


## Random vs chronological split

Training on earlier campaigns to predict later ones is the deployment scenario. A large drop here would mean the random split was flattering.

In [7]:
res[res.feature_set == "launch_only"][["split", "accuracy", "f1", "roc_auc"]].round(4)

,split,accuracy,f1,roc_auc
0,random,0.7153,0.7609,0.7798
2,chronological,0.7200,0.7588,0.7906


## Confusion matrix

The off-diagonal cells are two different harms: a **false positive** sends a creator into a campaign that fails, while a **false negative** discourages one who would have succeeded. Which matters more is a question Milestone 4 has to answer.

In [8]:
train, test = SPLITS["random"]
X_train, y_train = build_model_matrix(train, "launch_only", extra_numeric=EXTRA)
X_test, y_test = build_model_matrix(test, "launch_only", extra_numeric=EXTRA)
X_test = align_columns(X_train, X_test)

final = make_models()["Random Forest"]
final.fit(X_train, y_train)
confusion_frame(final, X_test, y_test)

,predicted: failed,predicted: successful
actual: failed,2014,1461
actual: successful,724,3477


## Save

Task 8 reads this file by name.

In [9]:
res.to_csv("../reports/m3_task7_random_forest.csv", index=False)
print("Wrote ../reports/m3_task7_random_forest.csv")

Wrote ../reports/m3_task7_random_forest.csv


## Summary

- Do **not** use the built-in `feature_importances_`. That is Gini
  importance, which is biased toward continuous and high-cardinality
  features. Task 9 uses permutation importance instead.
- `min_samples_leaf` is the main lever against overfitting. Raising it
  makes each tree simpler.
- More trees is more stable but with sharply diminishing returns past a
  few hundred.